# 第6周 · 无监督学习、梯度与文本基线

**本周结束时你能做到**：用 PCA 把 64 维的手写数字压到 2 维并解释"解释方差"，再用四步从零实现 PCA、用重建误差决定留几个主成分；用 k-means 聚类并用 ARI 打分；对任何函数亲手求导并用数值差分验证，包括手推逻辑回归的梯度（第7周 micrograd 的直接前置）；为 IMDB 影评情感分类搭起一座从朴素贝叶斯到 TF-IDF + 逻辑回归的**基线阶梯**——它是之后 TextCNN、Transformer、DistilBERT 的固定参照。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from zero2researcher import data
from zero2researcher.checks import week06 as ck

---
## S1 · 没有标签怎么办：PCA 与 k-means

前几周的模型都有"答案"（标签）可学。**无监督学习**在没有标签时找结构：

- **k-means 聚类**：把样本分成 k 群，每群围着自己的中心。
- **PCA（主成分分析）**：找数据方差最大的几个方向，把高维数据投影上去——压缩与可视化的标准工具。

先在一份能看见的二维合成数据上体会：

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

Xb, _ = make_blobs(n_samples=300, centers=4, cluster_std=1.0, random_state=42)
km4 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Xb)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].scatter(Xb[:, 0], Xb[:, 1], s=12, color="gray")
axes[0].set_title("What the algorithm sees: no labels")
axes[1].scatter(Xb[:, 0], Xb[:, 1], c=km4.labels_, s=12, cmap="tab10")
axes[1].scatter(km4.cluster_centers_[:, 0], km4.cluster_centers_[:, 1],
                marker="x", s=120, color="k")
axes[1].set_title("KMeans recovers the 4 clusters (x = centres)")
for a in axes:
    a.set_xlabel("x1")
    a.set_ylabel("x2")
plt.tight_layout()
plt.show()

### digits 的 PCA：64 维里到底有多少"真信息"？

第2周见过的手写数字：1797 张图、每张展平成 64 维。PCA 的 `explained_variance_ratio_` 告诉我们每个主成分方向"解释"了总方差的百分之几。

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

digits = load_digits()
Xd, yd = digits.data, digits.target

# 在这里写你的代码
raise NotImplementedError
print("前两个主成分的解释方差比:", np.round(evr[:2], 3))

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(np.arange(1, 65), np.cumsum(evr), ".-")
ax.set_xlabel("Number of components")
ax.set_ylabel("Cumulative explained variance")
ax.set_title("About 20 of 64 components keep ~90% of the variance")
plt.show()
ck.pca_evr(evr[:2])

In [ ]:
Z = PCA(n_components=2).fit_transform(Xd)
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(Z[:, 0], Z[:, 1], c=yd, cmap="tab10", s=8)
fig.colorbar(sc, label="True digit")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Two components already separate several digits")
plt.show()

只用 2/64 的维度，一些数字已经各占一角——而我们**全程没有用标签**（上色只是事后对答案）。

### 拆开 PCA：四步从零实现

`PCA().fit` 一行背后只有四步，每一步都是你已经会的东西：

1. **中心化**：每个像素减去自己的均值，`Xc = X - X.mean(axis=0)`；
2. **协方差矩阵**：`Xc.T @ Xc / (n - 1)`，形状 (64, 64)。第 4 周的 shape 规则：(64, n) @ (n, 64) → (64, 64)，第 i 行第 j 列是像素 i 与像素 j 的协方差；
3. **特征分解**：`np.linalg.eigh(cov)` 返回特征值和特征向量（每一**列**是一个特征向量）。特征向量就是主成分的方向，对应的特征值就是数据在这个方向上的方差（`eigh` 专门处理对称矩阵，协方差矩阵正好对称）；
4. **排序**：`eigh` 按特征值**从小到大**返回，要倒过来取前 k 个，特征向量跟着同一个顺序走。

写 `pca_from_scratch(X, k)`，返回 `(mean, components, explained_variance)`：`components` 的形状是 (k, 64)，**每一行**是一个主成分，与 sklearn 的 `components_` 一致。

In [ ]:
def pca_from_scratch(X, k):
    """返回 (mean, components, explained_variance)：components 每一行是一个主成分，按方差从大到小"""
    # 在这里写你的代码
    raise NotImplementedError


mean_d, comps_d, ev_d = pca_from_scratch(Xd, 64)
print("前两个主成分的解释方差比：从零实现", np.round(ev_d[:2] / ev_d.sum(), 4), "  sklearn", np.round(evr[:2], 4))
print(f"64 个方差与 sklearn 的最大差：{np.abs(ev_d - pca.explained_variance_).max():.1e}")
ck.pca_from_scratch(pca_from_scratch)

### 主成分的符号不唯一

把我们的主成分和 sklearn 的逐个做点积，会发现有的是 +1、有的是 −1。一个特征向量乘以 −1 仍是特征向量（所在的直线相同、指向相反），**两种都对**。sklearn 额外定了一条约定——让每个主成分里绝对值最大的那个分量为正；`eigh` 没有这条约定，给出哪个符号还取决于你电脑上的线性代数库。所以比较主成分时只能比"|点积| 是否等于 1"，不能直接比数值（自查也是这样做的）。ISLP 第 12 章的实验课把两种算法给出的结果描述为只差 "an unimportant sign flip"（一个无关紧要的符号翻转）。

对齐符号以后，两者就一致了。顺便验证另一条路线：对中心化后的数据做奇异值分解（SVD），奇异值的平方除以 n − 1，就是协方差矩阵的特征值（数据规模不同时，sklearn 会改走这条路）。

In [ ]:
dots = np.sum(comps_d * pca.components_, axis=1)           # 逐个主成分做点积
print("前 10 个主成分的点积:", np.round(dots[:10], 6))
print("前 20 个里符号相反的个数:", int((dots[:20] < 0).sum()), "（这个数随电脑而变，不要紧）")
aligned = comps_d * np.sign(dots)[:, None]                  # 把符号对齐到 sklearn 的约定
print(f"对齐符号后，前 20 个主成分的最大差：{np.abs(aligned[:20] - pca.components_[:20]).max():.1e}")

U, S, Vt = np.linalg.svd(Xd - mean_d, full_matrices=False)
print(f"SVD 路线：S² / (n − 1) 与特征值的最大差：{np.abs(S ** 2 / (len(Xd) - 1) - ev_d).max():.1e}")

### 重建误差：留几个主成分够用？

PCA 也是一种**压缩**：只存每张图在前 k 个主成分上的坐标（k 个数，而不是 64 个），需要时再"解压"回 64 维：

```
Z    = (X - mean) @ V.T        # 压缩：(n, 64) @ (64, k) → (n, k)
Xhat = Z @ V + mean            # 解压：(n, k) @ (k, 64) → (n, 64)
```

这里 V 是前 k 个主成分，形状 (k, 64)。解压后的图和原图差多少，就是**重建误差**（这里用每个像素的均方误差，像素取值 0–16）。k = 64 时没有任何损失；k 越小，丢的越多。在循环里写出重建那一行：

In [ ]:
ks = [1, 2, 5, 10, 20, 30, 40, 64]
Xc_d = Xd - mean_d
errs, first_digit = [], {}
for k in ks:
    V = comps_d[:k]                                    # 前 k 个主成分，(k, 64)
    # 在这里写你的代码
    raise NotImplementedError
    errs.append(float(np.mean((Xd - Xhat) ** 2)))
    first_digit[k] = Xhat[0]
print("重建误差（每像素 MSE）:", dict(zip(ks, np.round(errs, 3))))
print("舍弃的方差之和 × (n−1)/n ÷ 64:", np.round([(len(Xd) - 1) / len(Xd) * ev_d[k:].sum() / 64 for k in ks], 3))
ck.reconstruction_errors(ks, errs)

fig, axes = plt.subplots(1, 6, figsize=(9, 1.9))
axes[0].imshow(Xd[0].reshape(8, 8), cmap="gray_r")
axes[0].set_title("original", fontsize=8)
for ax, k in zip(axes[1:], [2, 5, 10, 20, 40]):
    ax.imshow(first_digit[k].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"k = {k}", fontsize=8)
for ax in axes:
    ax.axis("off")
fig.suptitle("The first digit, rebuilt from k principal components", fontsize=9)
plt.show()

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(ks, errs, "o-")
ax.set_xlabel("Number of components k")
ax.set_ylabel("Reconstruction MSE per pixel")
ax.set_title("Reconstruction error falls fast, then flattens")
plt.show()

两张图连起来读：k = 5 时数字已经大致能认出来，k = 20 时和原图几乎一样。上一格还打印了一个漂亮的恒等式：重建误差**恰好等于被舍弃的那些主成分的方差之和**（再乘 (n−1)/n、除以 64 个像素）。PCA 丢掉的正是方差最小的方向，所以在"只留 k 个方向"的所有线性压缩里，它的重建误差最小。

实践中常用累计解释方差来定 k：在 digits 上，保留 90% 的方差要 21 个主成分、95% 要 29 个，和上面曲线图标题里的"About 20 of 64"一致。

第 15 周会在 MNIST 上训练一个**线性自编码器**（一个"压缩—解压"的神经网络），按同样的每像素 MSE 和 PCA 比较，会发现它学到的正是 PCA 的子空间（Baldi & Hornik 1989）。

### k-means 聚 10 类，用 ARI 对答案

digits 恰好有 10 类，让 k-means 盲聚成 10 群，再和真实标签比。直接比准确率不行——聚类的"3 号簇"不一定对应数字 3。**ARI（调整兰德指数）** 解决了这一点：它看的是每一对样本："这两个在不在同一群"这件事，聚类结果和真实标签一致的程度（该在一起的在一起、该分开的分开），并扣掉了随机分群也能碰巧对上的部分——随机聚类约得 0，完美得 1。

**注意**：这里直接用原始像素（0–16），**不做标准化**——像素本来就同一量纲。

In [ ]:
from sklearn.metrics import adjusted_rand_score

# 在这里写你的代码
raise NotImplementedError
print("ARI =", round(ari, 3))
ck.kmeans_ari(ari)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(7, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(km10.cluster_centers_[i].reshape(8, 8), cmap="gray_r")
    ax.axis("off")
fig.suptitle("Cluster centres look like digits -- found without any labels")
plt.show()

> ⚠️ **一句话警示**：t-SNE 和 UMAP 的图只能用来"看一眼"——它们为了好看会扭曲几何，图上的**点间距离、簇的大小与间距都不可解读**，更不要在上面量距离做下游结论。

---
## S2 · 数学工作坊 2：导数、梯度与链式法则

第4周我们直接用了"MSE 对 w 的梯度"这个公式；第7周我们要从零搭出自动求导。今天把中间的数学补上。**导数**回答一个问题：输入动一点点，输出动多少？

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

右边不取极限、让 $h$ 取个很小的数（如 $10^{-5}$），就是**数值差分**。实际验证时我们用更准的**中心差分** $\frac{f(x+h) - f(x-h)}{2h}$：两侧各走一小步，误差比单侧小得多——本节每个手推结果都要用它验证。（小提醒：$|x|$、ReLU 这类函数在"尖角"处导数不存在，中心差分却照样会算出一个数，比如 ReLU 在 0 处得 0.5——那不是导数。）先把它写成函数：

In [ ]:
def numerical_gradient(f, x, h=1e-5):
    """数值差分估计 f 在 x 处的导数"""
    # 在这里写你的代码
    raise NotImplementedError

print(numerical_gradient(lambda x: x ** 2, 3.0))   # (x^2)' = 2x，在 x=3 处应约等于 6
ck.numerical_gradient(numerical_gradient)

### 复合函数与计算图

$f(x) = (3x + 2)^2$ 是两步运算的**复合**。把它画成计算图：

```
x ──(×3，+2)──▶ u ──(平方)──▶ f
```

**链式法则**：复合函数的导数 = 每一段导数的乘积。

$$\frac{df}{dx} = \frac{df}{du} \cdot \frac{du}{dx} = 2u \cdot 3$$

从输出端往输入端一段一段地乘回去——第7周的反向传播就是把这件事自动化。

In [ ]:
x = 1.0
u = 3 * x + 2        # 前向第一步：u = 5
f = u ** 2           # 前向第二步：f = 25

# 在这里写你的代码
raise NotImplementedError
print("链式法则手推:", df_dx)
print("数值差分验证:", numerical_gradient(lambda x: (3 * x + 2) ** 2, x))

### 偏导数与梯度向量

函数有多个输入时，**偏导数**是"只动这一个输入、冻住其他输入"的导数；把所有偏导数排成的向量叫**梯度**，它指向函数上升最快的方向（所以梯度下降往反方向走）。

对 $f(w_1, w_2) = w_1^2 + 3 w_2$：$\partial f / \partial w_1 = 2 w_1$，$\partial f / \partial w_2 = 3$。

In [ ]:
def f2(w1, w2):
    return w1 ** 2 + 3 * w2

w1, w2 = 2.0, -1.0
# 在这里写你的代码
raise NotImplementedError
numeric = np.array([numerical_gradient(lambda a: f2(a, w2), w1),
                    numerical_gradient(lambda b: f2(w1, b), w2)])
print("手推梯度:", grad)
print("数值验证:", numeric)

### 手推 MSE 对 w 的梯度（与第4周对上）

单参数线性模型 $\hat{y}_i = w x_i$，损失 $L(w) = \frac{1}{n}\sum_i (w x_i - y_i)^2$。

对每一项用链式法则：外层是平方（导数 $2(w x_i - y_i)$），内层 $w x_i - y_i$ 对 $w$ 的导数是 $x_i$。所以

$$\frac{dL}{dw} = \frac{1}{n}\sum_i 2 (w x_i - y_i) \, x_i$$

——这正是第4周 `gd_step` 里 `grad_w` 那一行（`2 / n * X.T @ (y_hat - y)`，只有一个特征时就是上式；第4周 S2 手算的那一步也是它）。验证：

In [ ]:
rng = np.random.default_rng(0)
x_data = rng.uniform(-1, 1, 50)
y_data = 2.5 * x_data + rng.normal(0, 0.1, 50)


def mse(w):
    return np.mean((w * x_data - y_data) ** 2)


w = 1.0
# 在这里写你的代码
raise NotImplementedError
print("手推梯度:", round(grad_w, 6))
print("数值差分:", round(numerical_gradient(mse, w), 6))

两个数字对上了。第7周的 micrograd 做的事：把"前向逐步计算、反向逐段乘导数"写成通用程序，让**任何**计算图都能自动得到梯度。

### 手推逻辑回归的梯度

第 5 周的逻辑回归是 sklearn 替我们训练的。现在有了链式法则，可以亲手推出它的梯度——推完你会发现，它和第 4 周 MSE 的梯度长得几乎一样。

对一个样本 (x, y)，y ∈ {0, 1}，计算图是三段：

```
x ──(z = w·x + b)──▶ z ──(p = σ(z))──▶ p ──(L = −[y log p + (1−y) log(1−p)])──▶ L
```

从输出端往回，一段一段地乘。前两段的导数：

1. $\dfrac{dL}{dp} = -\dfrac{y}{p} + \dfrac{1-y}{1-p} = \dfrac{p - y}{p(1-p)}$
2. $\dfrac{dp}{dz} = \sigma(z)\,\big(1 - \sigma(z)\big) = p(1-p)$（sigmoid 导数的一个漂亮性质，可以用数值差分验证）

剩下的几步留给你，**先在纸上推完**：

3. 把前两段乘起来，化简出 $\dfrac{dL}{dz}$；
4. 再乘上 $\dfrac{\partial z}{\partial w}$ 和 $\dfrac{\partial z}{\partial b}$，得到单个样本的梯度；
5. 损失是 n 个样本的**平均**交叉熵，梯度也要对样本取平均。写成向量化代码时对一对形状：`X` 是 (n, d)，把每个样本的 $\dfrac{dL}{dz}$ 排成一个 (n,) 的向量，`grad_w` 应是 (d,)——(d, n) @ (n,) → (d,)；`grad_b` 是一个数。

推完再写 `logistic_grad(X, y, w, b)`（`sigmoid` 在下一格开头给出），并用数值差分逐个坐标验证（推导的答案在下一格之后，写完再对照）：

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def logistic_grad(X, y, w, b):
    """平均交叉熵对 (w, b) 的梯度，返回 (grad_w, grad_b)"""
    # 在这里写你的代码
    raise NotImplementedError


def bce(X, y, w, b):
    """平均交叉熵"""
    p = sigmoid(X @ w + b)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


rng = np.random.default_rng(0)
X_lr = rng.normal(0, 1, size=(200, 2))
y_lr = rng.binomial(1, sigmoid(X_lr @ np.array([2.0, -1.0]) + 0.5))    # 造数据用的参数：w = (2, −1)，b = 0.5

w0, b0 = np.array([0.3, -0.2]), 0.1                                     # 在这一点上求梯度
gw, gb = logistic_grad(X_lr, y_lr, w0, b0)
num_w = [numerical_gradient(lambda t: bce(X_lr, y_lr, np.array([t, w0[1]]), b0), w0[0]),
         numerical_gradient(lambda t: bce(X_lr, y_lr, np.array([w0[0], t]), b0), w0[1])]
num_b = numerical_gradient(lambda t: bce(X_lr, y_lr, w0, t), b0)
print("手推梯度:", np.round(gw, 8), round(gb, 8))
print("数值差分:", np.round(num_w, 8), round(num_b, 8))
ck.logistic_grad(logistic_grad)

手推和数值差分对上了。对一对你的推导：

3. 两段相乘，分母正好约掉：$\dfrac{dL}{dz} = p - y$
4. $\dfrac{\partial z}{\partial w} = x$，$\dfrac{\partial z}{\partial b} = 1$，所以 $\dfrac{\partial L}{\partial w} = (p - y)\,x$，$\dfrac{\partial L}{\partial b} = p - y$
5. 对 n 个样本取平均，写成向量化形式：

```
grad_w = X.T @ (sigmoid(X @ w + b) - y) / n
grad_b = mean(sigmoid(X @ w + b) - y)
```

和第 4 周对照：MSE 的梯度是 `2/n * X.T @ (y_hat - y)`。形式都是"**误差 × 输入**"，只是逻辑回归的"误差"是预测概率减标签。

再看它和 sklearn 的关系：在 sklearn 训练出的最优解上，我们的梯度应该是 0。`C=np.inf` 表示**不加正则**（`penalty` 参数在 sklearn 1.8 起已弃用，见第 5 周 S2）。也可以用自己的梯度跑第 4 周的梯度下降循环，看看能不能走到同一个点：

In [ ]:
from sklearn.linear_model import LogisticRegression

clf_lr = LogisticRegression(C=np.inf, tol=1e-10, max_iter=10000).fit(X_lr, y_lr)     # C=np.inf：不加正则
gw_opt, gb_opt = logistic_grad(X_lr, y_lr, clf_lr.coef_[0], clf_lr.intercept_[0])
print("sklearn 的解: w =", np.round(clf_lr.coef_[0], 4), " b =", round(clf_lr.intercept_[0], 4))
print(f"在这一点上我们的梯度（绝对值最大的分量）：{np.abs(np.r_[gw_opt, gb_opt]).max():.1e}")

w_gd, b_gd = np.zeros(2), 0.0
for _ in range(5000):                       # 第 4 周的梯度下降循环，只是换了一个损失
    g_w, g_b = logistic_grad(X_lr, y_lr, w_gd, b_gd)
    w_gd, b_gd = w_gd - 1.0 * g_w, b_gd - 1.0 * g_b
print("梯度下降 5000 步: w =", np.round(w_gd, 4), " b =", round(b_gd, 4))

sklearn 的求解器停下的地方，梯度约为 1e-9：**梯度为 0 的地方就是最优解**，求解器做的正是"找梯度为 0 的点"；我们用自己的梯度做梯度下降，走到了同一个点。两点补充：

- 默认的 `C=1` 加了 L2 惩罚，最优点会被往 0 拉一点，在那里平均交叉熵的梯度就不再是 0——正则化改变了"梯度为 0"的位置（第 4 周的 Ridge 也是这样）。
- 与线性回归不同，逻辑回归令梯度为 0 解不出显式公式（没有"正规方程"），只能迭代求解。sklearn 默认的 L-BFGS 是一种拟牛顿法，会利用二阶导数（曲率）的近似走得更快；想深入可以读 [CS229 讲义](https://cs229.stanford.edu/main_notes.pdf) §2.4 的牛顿法，本课不展开。

---
## S3 · 文本主线启动：IMDB 影评情感分类

从今天起到第14周，我们反复使用同一份数据：IMDB 影评情感数据集（Maas et al. 2011）的**课程固定子集**——训练 5000 条、测试 2000 条，列 `text` 与 `label`（0 = 差评，1 = 好评），两类均衡、已去重并打乱。

In [ ]:
train = data.load_imdb("train")
test = data.load_imdb("test")
print(train.shape, test.shape)
print(train["label"].value_counts().to_dict())
print("\n一条好评的开头：\n", train.loc[train["label"] == 1, "text"].iloc[0][:300], "...")

### 把文字变成数字：词袋与 TF-IDF

模型只吃数字。最朴素的编码是**词袋**（bag of words）：数每个词出现了几次，语序全部丢掉。先在玩具语料上看 `CountVectorizer` 做了什么：

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

toy = ["a great great movie", "a terrible movie"]
v = CountVectorizer()
M = v.fit_transform(toy)
print(v.get_feature_names_out())
print(M.toarray())

**TF-IDF** 在词频基础上，把"哪篇都出现"的常见词（the、movie）压低权重，突出有区分度的词。两者都用 `min_df=2` 丢掉只在 1 条影评里出现的词（多是拼写错误和人名）。

### 基线阶梯

研究的第一铁律：**任何花哨模型，都必须先打败它脚下的阶梯**。今天搭的这座阶梯，就是第10周 TextCNN、第11周 Transformer、第14周 DistilBERT 的固定参照。

- 第 0 级：多数类基线（闭着眼睛猜**训练集**里最多的那类）
- 第 1 级：词袋 + 朴素贝叶斯
- 第 2 级：词袋 + 逻辑回归
- 第 3 级：TF-IDF + 逻辑回归
- 对照组：TF-IDF + 随机森林

In [ ]:
from sklearn.dummy import DummyClassifier

# 多数类基线：预测"训练集"里最多的那一类，再到测试集上打分——测试集不参与任何决定。
# （两类均衡时它和"测试集多数类的占比"相同；类别不均衡时两种写法会不一样，前者才不偷看测试集。）
dummy = DummyClassifier(strategy="most_frequent").fit(train["text"], train["label"])
majority_acc = dummy.score(test["text"], test["label"])
print("第 0 级 · 多数类基线:", majority_acc)

### 第 1 级：朴素贝叶斯——数一数每个词在好评、差评里各出现几次

**朴素贝叶斯**（Naive Bayes）可能是最简单的文本分类器：

1. 数一数：所有好评里，每个词一共出现了多少次；差评里也数一遍；
2. 换成频率：词 i 在好评里的概率 θᵢ = (该词在好评里的总次数 + α) / (好评的总词数 + α × 词表大小)。加 α（默认 1，叫**拉普拉斯平滑**）是为了让没在某一类里出现过的词概率不为 0；
3. 预测一条新影评：好评得分 = log P(好评) + Σ（这个词在影评里出现的次数 × log θᵢ(好评)），差评同理，取得分高的那一类。

"朴素"在于它假设：给定类别以后，每个词的出现互相独立。这显然不对（"not" 后面跟 "good" 意思就反了），但它照样能用。scikit-learn 文档对它的评价很到位：一个不错的分类器，一个糟糕的概率估计器。它只需要数数，训练几乎不花时间。

先建词袋特征（下一级的逻辑回归也用这一份，两级之间只换分类器），再训练 `MultinomialNB()`。向量化器叫 `bow`、特征矩阵叫 `X_bow`、模型叫 `nb`、测试准确率叫 `nb_acc`——后面几格会用到这些名字：

In [ ]:
from sklearn.naive_bayes import MultinomialNB

# 在这里写你的代码
raise NotImplementedError
print(f"第 1 级 · 词袋 + 朴素贝叶斯: {nb_acc:.4f}   （词表大小 {X_bow.shape[1]}）")
ck.nb_baseline(nb_acc)

In [ ]:
from sklearn.linear_model import LogisticRegression

# 在这里写你的代码
raise NotImplementedError
print(f"第 2 级 · 词袋 + 逻辑回归: {bow_acc:.4f}   （词表大小 {X_bow.shape[1]}）")

In [ ]:
# 在这里写你的代码
raise NotImplementedError
print(f"第 3 级 · TF-IDF + 逻辑回归: {tfidf_acc:.4f}")
ck.baseline_ladder(majority_acc, bow_acc, tfidf_acc)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 按课程设计限制到 2 万维。这个子集 min_df=2 的词表本来只有约 2.2 万维，限制只砍掉约 9% 的列，
# 速度和准确率都几乎不变；完整 IMDB（2.5 万条训练）的词表约 7.5 万维，那时限制维度才有实际意义。
tfidf20k = TfidfVectorizer(min_df=2, max_features=20000)
X_20k = tfidf20k.fit_transform(train["text"])
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_20k, train["label"])
rf_acc = rf.score(tfidf20k.transform(test["text"]), test["label"])
print(f"对照组 · TF-IDF + 随机森林: {rf_acc:.4f}")

In [ ]:
names = ["Majority", "BoW + NB", "BoW + LR", "TF-IDF + LR", "TF-IDF + RF"]
vals = [majority_acc, nb_acc, bow_acc, tfidf_acc, rf_acc]
fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar(names, vals, color=["gray", "C2", "C0", "C0", "C1"])
ax.bar_label(bars, fmt="%.3f")
ax.set_ylim(0.4, 1.0)
ax.set_ylabel("Test accuracy")
ax.set_title("Simple baselines are strong: TF-IDF + LR reaches ~0.87")
plt.show()

记住这几个数字：0.500 → 0.819 → 0.855 → 0.867。朴素贝叶斯只靠数数就拿到 0.82；同一份词袋特征换成逻辑回归，又多约 3.6 个百分点。之后任何"深度学习模型"如果没有明显越过 0.87 这一级，它就还没证明自己值那些算力。

顺便看看逻辑回归学到了什么——权重最大/最小的词：

In [ ]:
words = np.array(tfidf.get_feature_names_out())
w = clf_tfidf.coef_[0]
print("最'好评'的词:", words[np.argsort(w)[-10:][::-1]])
print("最'差评'的词:", words[np.argsort(w)[:10]])

再看朴素贝叶斯最"信"的词：按 log θ(好评) − log θ(差评) 排序，括号里是这个词出现在几条训练影评里。

In [ ]:
nb_words = np.array(bow.get_feature_names_out())
log_ratio = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]      # log θ(好评) − log θ(差评)
n_docs = np.asarray((X_bow > 0).sum(axis=0)).ravel()                # 每个词出现在几条影评里
for label, order in [("NB 最'好评'的词", np.argsort(log_ratio)[-10:][::-1]),
                     ("NB 最'差评'的词", np.argsort(log_ratio)[:10])]:
    print(label + ":", ", ".join(f"{nb_words[i]}({n_docs[i]})" for i in order))
common = np.where(n_docs >= 20)[0]                                  # 只看出现在 ≥ 20 条影评里的词
print("只看常见词，NB 最'好评'的:", nb_words[common[np.argsort(log_ratio[common])[-10:][::-1]]])
print("只看常见词，NB 最'差评'的:", nb_words[common[np.argsort(log_ratio[common])[:10]]])

两个模型最信的词很不一样。逻辑回归的权重词是 great、best、bad、worst 这类通用情感词；朴素贝叶斯排在最前面的却多是**人名和片名**（paulie、spock、seagal、boll……），大多只出现在十来条甚至更少的影评里，碰巧都落在同一类。朴素贝叶斯对每个词单独数数，几次碰巧的计数就能给出很大的 log 比值（拉普拉斯平滑 α = 1 是它唯一的"收缩"），于是把"少见、但碰巧只出现在一类里"的词当成了铁证；逻辑回归同时看所有词，又有 L2 正则（默认 `C=1`）把罕见词的权重往 0 拉。这是朴素贝叶斯在 5000 条影评上输给逻辑回归的原因之一。只看出现在 20 条以上影评里的词，名单就以 perfection、wonderfully、waste、unfunny 这类情感词为主了（仍夹着个别人名）。

作业 B 会看到故事的另一面：训练数据很少时，朴素贝叶斯反而赢。

---
## S4 · 研究方法微课：最小实验单元

从下周开始，你做的每一个对比都要长成下面这个样子——**最小实验单元**：

> **一个基线 + 一次受控修改 + 至少 3 个随机种子 + 均值 ± 标准差**

- **一次只改一个东西**（自变量），其余全部冻住（控制变量）；
- **至少 3 个种子**：单次运行的差异可能纯属运气，没有标准差的对比无法解读；
- 结论只能是三种之一：显著更好 / 显著更差 / **差异小于噪声**——第三种也是合法发现。

每做一个单元，就写一条**实验记录**（格式见 `templates/lab_notebook.md`）。现场示范——把今天 S3 的"词袋 → TF-IDF"写成一条：

---

## 实验记录 #0 · 日期：2026-10-30（示范）

- 研究问题：在 IMDB 课程子集上，TF-IDF 编码是否优于原始词频（词袋）？
- 假设：TF-IDF 压低常见词权重，测试准确率更高；若两者差距 < 1 个百分点则视为无实质差异。
- 自变量：文本编码方式（CountVectorizer → TfidfVectorizer）。
- 控制变量：同一训练/测试划分（课程固定子集）、`min_df=2`、`LogisticRegression(max_iter=1000)`。
- 基线：词袋 + 逻辑回归 = 0.8550。
- 数据与划分方式：课程固定子集 5000/2000，向量化器只在训练集上 fit（防泄漏）。
- 随机种子：无随机成分（划分固定、LBFGS 确定性）——**严格说本条只跑了"1 个种子"，不合最小单元标准**，作业里你将补上多种子版本。
- 结果：TF-IDF = 0.8665（+1.2 个百分点）。
- 解释：常见词权重被压低后，有区分度的情感词相对更突出。
- 局限：单一划分、单一超参数设定；提升幅度接近"1 个百分点"的判断线。
- 下一步：以 n-gram 范围为自变量、3 个训练子集为"种子"，做一次合格的最小实验单元（本周作业 A）。注意这样的"种子"只反映"换一批训练数据"的波动，不含测试集本身的抽样误差，会低估不确定性（作业 A 有详细说明）。

---

本周练习是一次**泄漏排查演练**（打开 `06_exercises.ipynb`）；作业 A 是你的**第一条正式实验记录**，之后所有研究作业都用这个格式记录。

### 本周小结

- PCA：找方差最大的方向；`explained_variance_ratio_` 告诉你留了多少信息
- PCA 四步：中心化 → 协方差 → `eigh` → 降序；主成分的符号不唯一（只比 |点积|）；重建误差 = 被舍弃的方差
- k-means：无标签聚类；与真实标签比较用 ARI；**预处理会改变结论**（digits 上标准化反而更差）
- t-SNE/UMAP 的图上距离不可解读
- 导数 = 输入动一点点输出动多少；链式法则 = 逐段导数相乘；一切手推都用数值差分验证
- 逻辑回归的梯度 = (σ(z) − y)·x：和 MSE 一样是"误差 × 输入"；最优解就是梯度为 0 的点
- 基线阶梯：多数类 0.500 → 词袋+朴素贝叶斯 0.819 → 词袋+LR 0.855 → TF-IDF+LR 0.867——深度模型的固定参照
- 最小实验单元：基线 + 一次受控修改 + ≥3 种子 + 均值±标准差，并写实验记录